# 01 - EDA and Hypothesis Tests

Interactive notebook for exploratory checks and statistical testing.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

PROJECT_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_preprocessing import load_data, basic_checks, handle_missing_values
from src.utils import TARGET_COLUMN, DIGITAL_DISTRACTION_FEATURES, OUTPUT_DIR

print('Project root:', PROJECT_ROOT)

In [ ]:
df, is_synthetic = load_data(project_root=PROJECT_ROOT)
print('Synthetic data used:' , is_synthetic)
basic_checks(df)
df = handle_missing_values(df, strategy='drop')
df.shape

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap (EDA)')
plt.tight_layout()
plt.show()

In [ ]:
test_cols = [
    'phone_usage_hours', 'social_media_hours', 'youtube_hours', 'gaming_hours',
    'study_hours_per_day', 'sleep_hours', 'exercise_minutes', 'focus_score'
]
rows = []
for col in test_cols:
    if col in df.columns:
        x = df[col].values
        y = df[TARGET_COLUMN].values
        pr, pp = pearsonr(x, y)
        sr, sp = spearmanr(x, y)
        rows.append({
            'feature': col,
            'pearson_r': pr,
            'pearson_p': pp,
            'spearman_rho': sr,
            'spearman_p': sp,
        })

hypothesis_df = pd.DataFrame(rows).sort_values('pearson_p')
hypothesis_df

In [ ]:
top = corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values()
plt.figure(figsize=(8, 6))
top.tail(12).plot(kind='barh', color='teal')
plt.title('Top Positive Correlations with Productivity')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

## Extra EDA checks (more detailed)

Below cells are intentionally more exploratory, for understanding what can go wrong later in modeling.

In [ ]:
# 1) quick data quality profile
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2),
    'n_unique': df.nunique()
}).sort_values(['missing_pct', 'n_unique'], ascending=[False, True])
quality.head(20)

In [ ]:
# 2) missing value profile chart
miss = (df.isnull().mean() * 100).sort_values(ascending=False)
plt.figure(figsize=(10, 4))
miss.plot(kind='bar', color='gray')
plt.title('Missing values by column (%)')
plt.ylabel('Percent')
plt.tight_layout()
plt.show()

In [ ]:
# 3) key distributions in one place
key = [
    'productivity_score', 'study_hours_per_day', 'sleep_hours', 'phone_usage_hours',
    'social_media_hours', 'youtube_hours', 'gaming_hours', 'stress_level', 'focus_score'
]
key = [c for c in key if c in df.columns]

n_cols = 3
n_rows = int(np.ceil(len(key) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = np.array(axes).reshape(-1)
for i, col in enumerate(key):
    axes[i].hist(df[col].dropna(), bins=25, edgecolor='black', alpha=0.75)
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# 4) outlier preview by IQR
outlier_rows = []
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COLUMN]
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    n_out = int(((df[col] < low) | (df[col] > high)).sum())
    outlier_rows.append({'feature': col, 'outlier_count': n_out, 'outlier_pct': 100 * n_out / len(df)})

outlier_df = pd.DataFrame(outlier_rows).sort_values('outlier_count', ascending=False)
outlier_df.head(15)

In [ ]:
# 5) distraction vs productivity (bucket view)
if set(DIGITAL_DISTRACTION_FEATURES).issubset(df.columns):
    tmp = df.copy()
    tmp['digital_distraction_score'] = tmp[DIGITAL_DISTRACTION_FEATURES].sum(axis=1)
    tmp['distraction_bucket'] = pd.qcut(
        tmp['digital_distraction_score'],
        q=4,
        labels=['Low', 'Med-Low', 'Med-High', 'High']
    )

    plt.figure(figsize=(8, 5))
    sns.boxplot(data=tmp, x='distraction_bucket', y=TARGET_COLUMN)
    plt.title('Productivity by digital distraction bucket')
    plt.tight_layout()
    plt.show()

    tmp.groupby('distraction_bucket')[TARGET_COLUMN].describe()[['mean', 'std', 'min', 'max']]

In [ ]:
# 6) pairplot sample (small sample so notebook dont explode)
pair_cols = [
    TARGET_COLUMN, 'study_hours_per_day', 'sleep_hours',
    'phone_usage_hours', 'social_media_hours', 'focus_score'
]
pair_cols = [c for c in pair_cols if c in df.columns]
if len(pair_cols) >= 3:
    sample_n = min(250, len(df))
    pair_df = df[pair_cols].sample(sample_n, random_state=42)
    g = sns.pairplot(pair_df, corner=True, plot_kws={'alpha': 0.5, 's': 18})
    g.fig.suptitle('Pairplot sample (key vars)', y=1.02)
    plt.show()

In [ ]:
# 7) extra hypothesis test style checks (split-based)
# Not causal proof, just statistical difference checks.
from scipy.stats import ttest_ind

if 'gender' in df.columns:
    gvals = df['gender'].astype(str).str.title()
    groups = [x for x in gvals.unique() if x == x]
    if len(groups) >= 2:
        g1 = df.loc[gvals == groups[0], TARGET_COLUMN].dropna()
        g2 = df.loc[gvals == groups[1], TARGET_COLUMN].dropna()
        stat, p = ttest_ind(g1, g2, equal_var=False)
        print(f't-test productivity by gender: {groups[0]} vs {groups[1]} -> t={stat:.3f}, p={p:.4g}')

if set(DIGITAL_DISTRACTION_FEATURES).issubset(df.columns):
    score = df[DIGITAL_DISTRACTION_FEATURES].sum(axis=1)
    q1 = score.quantile(0.25)
    q3 = score.quantile(0.75)
    low = df.loc[score <= q1, TARGET_COLUMN].dropna()
    high = df.loc[score >= q3, TARGET_COLUMN].dropna()
    stat, p = ttest_ind(low, high, equal_var=False)
    print(f't-test low vs high distraction quartiles -> t={stat:.3f}, p={p:.4g}')

In [ ]:
# 8) save key test table for report drafting
hypothesis_df.to_csv(os.path.join(PROJECT_ROOT, 'outputs', 'notebook_hypothesis_tests.csv'), index=False)
print('saved -> outputs/notebook_hypothesis_tests.csv')